In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

ospath = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(ospath)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
df = df.fillna(df.mean())

In [ ]:
# Task 2: Write your code here:
# 4. Check for duplicates
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df.head(-1)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
df


In [ ]:
# Task 4: Write your code here:
numerical_cols = df.select_dtypes(include=["number"]).columns.drop('Target')  # DON'T SCALE THE TARGET   # or include = ["number"]

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df['Target'].hist(bins=20)
  plt.show()

check_target_imbalance(df, 'Target')
# target is imbalanced littel  we might take the log of the target

In [ ]:
# Task 1: Write your code here:
X = df.drop('Target', axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
%pip install  catboost l -q

In [ ]:
# Task 2,3,4,5: Write your code here:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr_accuracy = []
lr_f1 = []
# 5-Fold Cross-Validation
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

model = CatBoostClassifier(verbose=0,
      n_estimators=320,
      max_depth=4)
model.fit(X_train, y_train)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_f1.append(f1)


print(f"5-Fold CV Results:")
print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"  F1-Score:  {np.mean(lr_f1):.4f}")


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': numerical_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print(f"your golden feature, the most important feature is {feature_importance['feature'][0]} \n enjoy(:")

In [ ]:
# Task Bonus: Write your code here:
X_train=X_train['P_2']
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train['P_2'], X_test['P_2'] = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_f1.append(f1)


print(f"5-Fold CV Results:")
print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"  F1-Score:  {np.mean(lr_f1):.4f}")
